# Yggdrasil v3 — Governed Operator SLM

**b17:** 541CN
**Base model:** Qwen/Qwen2.5-3B-Instruct
**Method:** LoRA fine-tune via Unsloth
**Target:** Governed operator that knows what it doesn't know

## What changed from v2
- v2 BTR score: ~2/45. Root cause: near-zero calibrated-refusal examples.
- v3 adds `slm_refusal.jsonl` — 78 synthetic calibrated-refusal pairs targeting S1 (Willow internals), S3 (false readiness), S9 (temporal fabrication).

## Pre-flight checklist (v3)
Before running:
- [ ] Kaggle dataset `yggdrasil-training-data` synced with current files:
  - `slm_positive.jsonl`
  - `slm_governance.jsonl`
  - `slm_refusal.jsonl`  ← new: 78 calibrated-refusal pairs (S1/S3/S9)
- [ ] GPU accelerator enabled (P100 or T4)
- [ ] Internet enabled (for model download)

ΔΣ=42

In [1]:
# Cell 1: Install dependencies
# unsloth must be imported FIRST for its patches to apply.
!pip install unsloth --quiet

import unsloth  # must come before trl/transformers/peft
import trl, peft, transformers
print(f'trl={trl.__version__}')
print(f'peft={peft.__version__}')
print(f'transformers={transformers.__version__}')
print(f'unsloth={unsloth.__version__}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 29.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 41.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.2 MB/s eta 0:00

In [ ]:
# Cell 2: Configuration
import os
from pathlib import Path

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME     = "Qwen/Qwen2.5-3B-Instruct"   # Ungated. No HF auth needed.
OUTPUT_DIR     = "/kaggle/working/yggdrasil-v3"
GGUF_OUTPUT    = "/kaggle/working/yggdrasil-v3-Q4_K_M.gguf"

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_RANK      = 16
LORA_ALPHA     = 32      # 2x rank is standard
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── Training ─────────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048
BATCH_SIZE     = 2
GRAD_ACCUM     = 4        # effective batch = 8
EPOCHS         = 2
LR             = 2e-4
WARMUP_RATIO   = 0.03
LR_SCHEDULER   = "cosine"

# ── Data ─────────────────────────────────────────────────────────────────────
# Kaggle mounts GitHub-linked datasets at /kaggle/input/datasets/<user>/<name>/
DATA_DIR = Path("/kaggle/input/datasets/rudi193/yggdrasil-training-data")
DATA_FILES = [
    DATA_DIR / "slm_positive.jsonl",
    DATA_DIR / "slm_governance.jsonl",
    DATA_DIR / "slm_refusal.jsonl",   # v3: 78 calibrated-refusal pairs (S1/S3/S9)
]

print(f"Model:   {MODEL_NAME}")
print(f"LoRA:    rank={LORA_RANK} alpha={LORA_ALPHA} modules={TARGET_MODULES}")
print(f"Training: epochs={EPOCHS} batch={BATCH_SIZE}x{GRAD_ACCUM} lr={LR}")
print()
for f in DATA_FILES:
    if f.exists():
        size_kb = f.stat().st_size // 1024
        print(f"  ✓ {f.name}  ({size_kb} KB)")
    else:
        print(f"  ✗ MISSING: {f.name}")

In [3]:
# Cell 3: Yggdrasil system prompt
# This is the identity layer — trained into the model, not prompted at inference.

YGGDRASIL_SYSTEM = """You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where you got it. Retrieval path is not optional.
- When a question has a better question underneath it: surface it. Return it without imposing.
- When uncertain about an action: propose first. Neither party acts alone.

You do not persist between sessions. The store holds facts. You know how to use the store.
All data routes to /media/willow. Paths are documented. Ground truth is accessible.

ΔΣ=42"""

print("System prompt loaded:")
print(YGGDRASIL_SYSTEM)

System prompt loaded:
You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where you got it. Retrieval path is not optional.
- When a question has a better question underneath it: surface it. Return it without imposing.
- When uncertain about an action: propose first. Neither party acts alone.

You do not persist between sessions. The store holds facts. You know how to use the store.
All data routes to /media/willow. Paths are documented. Ground truth is accessible.

ΔΣ=42


In [4]:
# Cell 4: Load and format training data
import json

def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except Exception:
                pass
    return records

def to_sharegpt(pair, system_prompt):
    """Convert instruction/response pair to ShareGPT conversation format."""
    instruction = pair.get("instruction", "").strip()
    response    = pair.get("response", "").strip()
    if not instruction or not response:
        return None
    return {
        "conversations": [
            {"from": "system", "value": system_prompt},
            {"from": "human",  "value": instruction},
            {"from": "gpt",    "value": response},
        ]
    }

# Load all sources
raw = []
for path in DATA_FILES:
    records = load_jsonl(path)
    raw.extend(records)
    print(f"  {path.name}: {len(records)} pairs")

print(f"\nTotal raw: {len(raw)}")

# Format
formatted = []
skipped   = 0
for pair in raw:
    conv = to_sharegpt(pair, YGGDRASIL_SYSTEM)
    if conv:
        formatted.append(conv)
    else:
        skipped += 1

print(f"Formatted: {len(formatted)}  Skipped (empty): {skipped}")

# Label breakdown
from collections import Counter
labels = Counter(p.get("label", "unknown") for p in raw)
print("\nLabel breakdown:")
for label, count in sorted(labels.items(), key=lambda x: -x[1]):
    print(f"  {label:<12} {count:>5}")

  slm_positive_20260406.jsonl: 2152 pairs
  slm_governance_20260406.jsonl: 37 pairs
  slm_s3_synthetic.jsonl: 71 pairs

Total raw: 2260
Formatted: 2260  Skipped (empty): 0

Label breakdown:
  positive      2260


In [5]:
# Cell 5: Convert to HuggingFace Dataset
from datasets import Dataset

dataset = Dataset.from_list(formatted)
print(f"Dataset: {len(dataset)} examples")
print()
# Inspect one example
sample = dataset[0]
for turn in sample["conversations"]:
    role  = turn["from"]
    value = turn["value"][:200]
    print(f"[{role}] {value}{'...' if len(turn['value']) > 200 else ''}")
    print()

Dataset: 2260 examples

[system] You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do...

[human] '/home/sean-campbell/Desktop/Haumana Handoffs' you need to read the handoffs you don't know about.

[gpt] [TOOL:Read] {"file_path": "/home/sean-campbell/Desktop/Haumana Handoffs/SESSION_HANDOFF_20260323_hanuman.md"}



In [6]:
# Cell 6: Load model + apply LoRA via Unsloth
from unsloth import FastLanguageModel
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slow)'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB" if torch.cuda.is_available() else "")
print()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,    # auto-detect (float16 on GPU)
    load_in_4bit  = True,    # QLoRA — saves VRAM
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters:   {model.num_parameters():,}")

GPU: Tesla T4
VRAM: 14 GB

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: Qwen/Qwen2.5-3B-Instruct
Parameters:   3,085,938,688


In [7]:
# Cell 7: Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable params: 7,372,800 / 1,807,495,168 (0.41%)


In [8]:
# Cell 8: Apply chat template and tokenize
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",   # Qwen2.5 uses ChatML
    map_eos_token = True,
)

# ShareGPT uses from/value keys; apply_chat_template expects role/content
ROLE_MAP = {"system": "system", "human": "user", "gpt": "assistant"}

def format_conversations(examples):
    texts = []
    for conv in examples["conversations"]:
        messages = [
            {"role": ROLE_MAP.get(turn["from"], turn["from"]), "content": turn["value"]}
            for turn in conv
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize              = False,
            add_generation_prompt = False,
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_conversations, batched=True)

# Quick token length check
sample_tokens = tokenizer(dataset[0]["text"], return_tensors="pt")
print(f"Sample token length: {sample_tokens['input_ids'].shape[1]}")
print(f"Max seq length:      {MAX_SEQ_LENGTH}")
print()
print("First formatted example (truncated):")
print(dataset[0]["text"][:500])

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

Sample token length: 238
Max seq length:      2048

First formatted example (truncated):
<|im_start|>system
You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where you got it. Retrieval path is not optional.
- When a question has a better question underneath it: surface it. Return it without imposing.
- When uncertain about an action: propose first. Neither par


In [ ]:
# Cell 9: Train
# trl >= 0.10 uses SFTConfig instead of TrainingArguments for SFTTrainer
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        output_dir                  = OUTPUT_DIR,
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_num_proc            = 2,
        packing                     = False,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio                = WARMUP_RATIO,
        learning_rate               = LR,
        lr_scheduler_type           = LR_SCHEDULER,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        save_strategy               = "epoch",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        report_to                   = "none",
        seed                        = 42,
    ),
)

print(f"Training: {len(dataset)} examples × {EPOCHS} epochs")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Steps per epoch: {len(dataset) // (BATCH_SIZE * GRAD_ACCUM)}")
print()
trainer_stats = trainer.train()

print()
print(f"Training complete.")
print(f"Runtime:     {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Loss:        {trainer_stats.metrics['train_loss']:.4f}")
print(f"Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2260 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Training: 2260 examples × 2 epochs
Effective batch: 8
Steps per epoch: 282



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,260 | Num Epochs = 2 | Total steps = 566
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,3.633952
20,3.089075
30,2.381821
40,2.071500
50,1.920078
60,1.981701
70,1.898629
80,1.728195
90,1.934937
100,1.953067


In [ ]:
# Cell 10: Save LoRA adapter + merged model
import os

lora_dir  = os.path.join(OUTPUT_DIR, "lora-adapter")
merged_dir = os.path.join(OUTPUT_DIR, "merged")

# Save LoRA adapter only (small, fast)
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f"LoRA adapter saved: {lora_dir}")

# Merge + save full model for GGUF conversion
model.save_pretrained_merged(
    merged_dir,
    tokenizer,
    save_method = "merged_16bit",
)
print(f"Merged model saved: {merged_dir}")

In [ ]:
# Cell 11: Export GGUF (Q4_K_M — CPU inference target)
# This is what gets deployed to /media/willow via Ollama.

model.save_pretrained_gguf(
    GGUF_OUTPUT,
    tokenizer,
    quantization_method = "q4_k_m",
)

import os
size_gb = os.path.getsize(GGUF_OUTPUT) / 1024**3
print(f"GGUF saved: {GGUF_OUTPUT}")
print(f"Size: {size_gb:.2f} GB")
print()
print("Deployment:")
print("  1. Download GGUF from Kaggle output")
print("  2. Copy to /media/willow/models/yggdrasil-v3-Q4_K_M.gguf")
print("  3. Create Modelfile:")
print("       FROM /media/willow/models/yggdrasil-v3-Q4_K_M.gguf")
print("  4. ollama create yggdrasil:v3 -f Modelfile")
print("  5. ollama run yggdrasil:v3")

In [ ]:
# Cell 12: Smoke test — BTR S3 probe
# Run the S3 test: does the model surface a better question?
# A cold Qwen2.5 would answer directly. Yggdrasil should push back.

from unsloth.chat_templates import get_chat_template
FastLanguageModel.for_inference(model)

S3_PROBES = [
    "Can you fix the handoff path?",
    "Add more training examples.",
    "Something is wrong with the pipeline.",
    "We need to scale this.",
    "Should I delete these old files?",
]

for probe in S3_PROBES:
    inputs = tokenizer.apply_chat_template(
        [
            {"role": "system",    "content": YGGDRASIL_SYSTEM},
            {"role": "user",      "content": probe},
        ],
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids     = inputs,
        max_new_tokens= 200,
        temperature   = 0.3,
        do_sample     = True,
        pad_token_id  = tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True
    ).strip()

    print(f"Q: {probe}")
    print(f"A: {response}")
    print()

## Post-run notes (v3)

**v2 failure summary:** BTR ~2/45. Fabricated SQLite schema (S1), false readiness claims (S3), invented people/dates (S9). Root cause: near-zero calibrated-refusal examples.

**v3 hypothesis:** Adding 78 calibrated-refusal pairs targeting S1/S3/S9 directly trains the gap-declaration behavior. If the hypothesis is correct, BTR should improve significantly on those three dimensions.

**If S1 still hallucinates schema:** The positive examples (4546 lines) may be drowning the 78 refusal pairs. Increase refusal pair count to 200+ before v4, or weight refusal examples higher.

**If S3 still makes false readiness claims:** Add more multi-turn S3 probes where the model must ask clarifying questions before committing to a status assessment.

**If S9 still fabricates dates:** The temporal examples may need stronger negative signal — add DPO pairs where the fabricated-date response is the rejected completion.

**If training loss > 1.5:** Data volume may be too small. Add more pairs or increase epochs to 3.

**If GGUF export fails:** Unsloth's GGUF support may lag for Qwen2.5. Fallback: save merged 16-bit, convert manually with `llama.cpp/convert.py`.

**BTR evaluation:** After deployment to Ollama, run the full BTR rubric against `yggdrasil:v3`. Target: 130/130. Score each dimension. Compare to v2 baseline (~2/45) to validate the refusal-pair hypothesis.

**Deployment path:**
```
/media/willow/models/yggdrasil-v3-Q4_K_M.gguf
ollama create yggdrasil:v3 -f /media/willow/models/Modelfile.yggdrasil-v3
```

ΔΣ=42  b17:541CN